In [ ]:
import glob
import tqdm

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.patches import Patch

from utils import *

BASE_PATH = '../IS26-experiments/'

In [ ]:
aligns = []
for aligns_path in tqdm.tqdm(glob.glob(f'{BASE_PATH}/**/aligns.pkl', recursive=True)):
    df = pd.read_pickle(aligns_path)
    experiment_config = read_config_from_path(aligns_path, BASE_PATH)
    for column, value in experiment_config.items():
        df[column] = value
    aligns.append(df)

if len(aligns) == 0:
    raise Warning(f"No aligns found in the specified path: {BASE_PATH}")

aligns = pd.concat(aligns)
aligns = aligns.reset_index(drop=True)

In [ ]:
aligners = aligns['aligner'].str.replace('-non_speech', '').str.split('-postprocess-').str[0]
root_common_tokens = get_common_tokens_with_root(list(aligners.unique()))
aligns['aligner_name'] = aligners.map(lambda name: compute_short_name_with_root(name, root_common_tokens))

postprocess_aligner = aligns['aligner'].apply(lambda x: x.split('-postprocess-')[1] if '-postprocess-' in x else None)
aligns['postprocess_aligner_name'] = postprocess_aligner.map(lambda name: compute_short_name_with_root(name, root_common_tokens) if pd.notna(name) else None)
aligns['short_aligner'] = aligns.apply(lambda row: row['aligner_name'] if pd.isna(row['postprocess_aligner_name']) else f"{row['aligner_name']}\n+\n{row['postprocess_aligner_name']}", axis=1)
aligns['intervals_condition'] = aligns['aligner'].apply(lambda x: 'non_speech' if 'non_speech' in x else 'full_audio' if 'full_audio' in x else 'only_speech')  

aligns['interval_duration'] = aligns['end'] - aligns['start']

In [ ]:
audio_durations = aligns[(aligns['aligner'] == 'full_audio') & (aligns['subset'] == 'original')].groupby('sample_id').first()['interval_duration'].reset_index().set_index('sample_id')
audio_durations.columns = ['audio_duration']
audio_durations.head()

In [ ]:
for dataset, dataset_df in aligns.groupby('dataset'):
    fig, axes = plt.subplots(figsize=(20, 4 * len(dataset_df.subset.unique())), nrows=len(dataset_df.subset.unique()), ncols=1, sharey=True)
    if len(dataset_df.subset.unique()) == 1: axes = [axes]    
    only_non_speech_intervals = dataset_df[dataset_df.intervals_condition == 'non_speech'].copy()
    for ax, (subset, subset_df) in zip(axes, only_non_speech_intervals.groupby('subset')):
        subset_df.boxplot(column='interval_duration', by='short_aligner', ax=ax, grid=False)
        ax.set_title(f'Subset: {subset} ({subset_df.sample_id.nunique()} samples)')
        ax.set_ylabel('Non-speech interval duration (s)')
        ax.set_xlabel('Aligner')
    plt.suptitle(f'Distribution of interval durations by aligner only for segments without speech\nDATASET: {dataset.upper()}')
    plt.tight_layout()
    plt.show()

In [ ]:
for dataset, dataset_df in aligns.groupby('dataset'):
    fig, axes = plt.subplots(figsize=(20, 4 * len(dataset_df.subset.unique())), nrows=len(dataset_df.subset.unique()), ncols=1, sharey=True)
    if len(dataset_df.subset.unique()) == 1: axes = [axes]
    only_non_speech_intervals = dataset_df[dataset_df.intervals_condition == 'non_speech'].copy()
    for ax, (subset, subset_df) in zip(axes, only_non_speech_intervals.groupby('subset')):
        subset_df.groupby(['short_aligner', 'sample_id'])['interval_duration'].sum().reset_index().boxplot(column='interval_duration', by='short_aligner', ax=ax, grid=False)
        ax.set_title(f'Subset: {subset} ({subset_df.sample_id.nunique()} samples)')
        ax.set_ylabel('Total non-speech duration per sample (s)')
        ax.set_xlabel('Aligner')
    plt.suptitle(f'Distribution of total non-speech duration per sample\nDATASET: {dataset.upper()}')
    plt.tight_layout()
    plt.show()

In [ ]:
only_non_speech_intervals = aligns[aligns.intervals_condition == 'non_speech'].copy()
only_non_speech_intervals = only_non_speech_intervals.groupby(['dataset', 'subset', 'short_aligner', 'sample_id'])['interval_duration'].sum().reset_index()
only_non_speech_intervals.groupby(['dataset', 'subset', 'short_aligner'])['interval_duration'].describe()

In [ ]:
def flatten_intervals(df):
    if df.empty:
        return []
    intervals = sorted(list(zip(df.start, df.end)), key=lambda x: x[0])
    merged = [intervals[0]]
    for current_start, current_end in intervals[1:]:
        prev_start, prev_end = merged[-1]
        if current_start < prev_end: 
            merged[-1] = (prev_start, max(prev_end, current_end))
        else:
            merged.append((current_start, current_end))
    return merged

def duration_of_intervals(intervals):
    return sum(end - start for start, end in intervals)


In [ ]:
manual_per_dataset = {
    'spanishad': 'manual'
}

manual_name_for_speech_segments = manual_per_dataset[dataset]
gt_only_speech = aligns[(aligns.short_aligner == manual_name_for_speech_segments) & 
                        (aligns.intervals_condition != 'non_speech')].drop_duplicates(subset=['sample_id', 'start', 'end'], keep='first')
gt_non_speech = aligns[(aligns.short_aligner == manual_name_for_speech_segments) & 
                       (aligns.intervals_condition == 'non_speech')].drop_duplicates(subset=['sample_id', 'start', 'end'], keep='first')

In [ ]:
mismatch_percentages = []
for (dataset, subset), dataset_df in aligns.groupby(['dataset', 'subset']):
    if dataset not in manual_per_dataset.keys():
        continue
    
    manual_name_for_speech_segments = manual_per_dataset[dataset]
    for sample_id, sample_aligns in dataset_df.groupby('sample_id'):
        
        non_speech_aligns_gt = gt_non_speech[gt_non_speech.sample_id == sample_id]
        non_speech_duration_gt = duration_of_intervals(non_speech_aligns_gt[['start', 'end']].values)
        
        speech_aligns_gt = gt_only_speech[gt_only_speech.sample_id == sample_id]
        speech_duration_gt = duration_of_intervals(speech_aligns_gt[['start', 'end']].values)
                
        sample_duration = audio_durations.loc[sample_id]['audio_duration']
        if sample_duration == speech_duration_gt + non_speech_duration_gt:
            print(f"Sample {sample_id} - Duration mismatch: audio={sample_duration}, speech={speech_duration_gt}, non-speech={non_speech_duration_gt}")
        
        for i, (short_aligner, non_speech_vad_aligner) in enumerate(sample_aligns.groupby('short_aligner'), start=1):
            if short_aligner == manual_name_for_speech_segments:
                continue
            non_speech_vad_aligners = non_speech_vad_aligner[non_speech_vad_aligner.intervals_condition == 'non_speech']
            non_speech_vad_duration = non_speech_vad_aligners.interval_duration.sum()
            if non_speech_vad_duration == 0:
                proportion_leakage_speech = 0.0
                proportion_missed_non_speech = 1
            else:
                speech_leakage_duration = 0.0
                for _, sil_row in non_speech_vad_aligners.iterrows():
                    for _, speech_row in speech_aligns_gt.iterrows():
                        start_inter = max(sil_row.start, speech_row.start)
                        end_inter = min(sil_row.end, speech_row.end)
                        if start_inter < end_inter:
                            speech_leakage_duration += end_inter - start_inter
                proportion_leakage_speech = (speech_leakage_duration / speech_duration_gt) * 100

                correct_silence_duration = 0.0
                for _, non_speech_vad_row in non_speech_vad_aligners.iterrows():
                    for _, non_speech_gt_row in non_speech_aligns_gt.iterrows():
                        start_inter = max(non_speech_vad_row.start, non_speech_gt_row.start)
                        end_inter = min(non_speech_vad_row.end, non_speech_gt_row.end)
                        if start_inter < end_inter:
                            correct_silence_duration += end_inter - start_inter
                proportion_missed_non_speech = ((non_speech_duration_gt - correct_silence_duration) / non_speech_duration_gt) * 100

            mismatch_percentages.append({
                'dataset': dataset,
                'subset': subset,
                'sample_id': sample_id,
                'short_aligner': short_aligner,
                'proportion_speech_leakage': proportion_leakage_speech,
                'proportion_missed_non_speech': proportion_missed_non_speech
            })
        
mismatch_percentages = pd.DataFrame(mismatch_percentages)
mismatch_percentages['label'] = mismatch_percentages['subset'] + '\n' + mismatch_percentages['short_aligner']
mismatch_percentages = mismatch_percentages.reset_index(drop=True)


In [ ]:
def plot_distribution(df, col, title):
    order = (df.groupby("label")[col].median().sort_values().index)
    plt.figure(figsize=(20, 6))

    sns.boxplot(data=df, x="label", y=col, order=order, color='lightblue', showfliers=False)
    sns.stripplot(data=df, x="label", y=col, order=order, alpha=0.8, size=3, color='grey', jitter=True)
    plt.xlabel("Subset + Aligner")
    plt.ylim(0, df[col].quantile(0.95))  
    plt.title(title)
    plt.tight_layout()
    plt.show()
    

In [ ]:
print("Speech leakage into non-speech segments (%) — summary (lower is better)")
plot_distribution(mismatch_percentages, 'proportion_speech_leakage', 'Proportion of speech leakage into non-speech segments (%) by aligner and subset\n(lower is better)')

mismatch_percentages.groupby(['subset', 'short_aligner'])['proportion_speech_leakage'].agg(['mean', 'std']).reset_index().sort_values('mean')

In [ ]:
plot_distribution(mismatch_percentages, 'proportion_missed_non_speech', 'Non-speech duration vs manual (detected - manual)')

print("Non-speech duration vs manual (detected - manual) — summary (negative: consider speech as non-speech segments; positive: missed non-speech segments):")
mismatch_percentages.groupby(['dataset', 'subset', 'short_aligner'])['proportion_missed_non_speech'].agg(['mean', 'std']).reset_index().sort_values('mean')


In [ ]:
speech = '#4C72B0'
no_speech = "#989696"
highlight_color = '#C44E52'

In [ ]:
for dataset_name, manual_aligner in manual_per_dataset.items():
    dataset_aligns = aligns[aligns.dataset == dataset_name]
    unique_case = dataset_aligns.sample(1, random_state=28).iloc[0].sample_id
    
    manual_name_for_speech_segments = manual_per_dataset[dataset_name]
    manual_only_speech_case = dataset_aligns[(dataset_aligns.short_aligner == manual_name_for_speech_segments)  & (dataset_aligns.intervals_condition != 'non_speech') & (dataset_aligns.sample_id == unique_case)].drop_duplicates(subset=['start', 'end'], keep='first')
    non_speech_duration_gt = (manual_only_speech_case.end - manual_only_speech_case.start).sum()

    for subset_name, subset_aligns in dataset_aligns.groupby('subset'):
        fig, ax = plt.subplots(figsize=(15, 0.5 * len(subset_aligns.short_aligner.unique()) + 2))
        yticks_locs, yticks_labels = [], []
        
        proportion_pos = subset_aligns[subset_aligns.sample_id == unique_case].end.max() + 0.4
        ax.barh(y=[0]*len(manual_only_speech_case), width=manual_only_speech_case.end - manual_only_speech_case.start, 
                left=manual_only_speech_case.start, height=0.6, color=[speech]*len(manual_only_speech_case), 
                edgecolor='white', alpha=0.6)
        yticks_locs.append(0)
        yticks_labels.append('Speech Manual Alignment (Reference)')
            
        subset = subset_aligns[(subset_aligns.sample_id == unique_case) & (subset_aligns.intervals_condition == 'non_speech')]
        for i, (aligner_name, non_speech_vad_aligner) in enumerate(subset.groupby('short_aligner'), start=1):
            widths = non_speech_vad_aligner.end - non_speech_vad_aligner.start
            
            ax.barh(y=[i]*len(non_speech_vad_aligner), width=widths, left=non_speech_vad_aligner.start, height=0.6, 
                    color=[no_speech]*len(non_speech_vad_aligner), edgecolor='white', alpha=0.6)
            
            speech_leakage_duration = 0.0
            if aligner_name != manual_name_for_speech_segments:
                for _, sil_row in non_speech_vad_aligner.iterrows():
                    for _, speech_row in manual_only_speech_case.iterrows():
                        start_inter = max(sil_row.start, speech_row.start)
                        end_inter = min(sil_row.end, speech_row.end)
                        if start_inter < end_inter:
                            dur = end_inter - start_inter
                            speech_leakage_duration += dur
                            ax.barh(y=i, width=end_inter - start_inter, left=start_inter, height=0.3, color=highlight_color, zorder=10)
            
            proportion = (speech_leakage_duration / non_speech_duration_gt) * 100 if non_speech_duration_gt > 0 else 0
            proportion = int(proportion) if int(proportion) == round(proportion, 1) else round(proportion, 1)
            ax.text(proportion_pos, i, f'{proportion}%', va='center', ha='left', fontweight='bold', color=highlight_color)
            yticks_locs.append(i)
            yticks_labels.append(aligner_name)

        ax.set_yticks(yticks_locs)
        ax.set_yticklabels(yticks_labels)
        ax.set_xlabel('Time (s)')
        ax.set_title(f'Alignment Discrepancies\nSAMPLE ID: {unique_case} - DATASET: {dataset_name.upper()} - SUBSET: {subset_name}')

        legend_elements = [
            Patch(facecolor=speech, label='Speech', alpha=0.6),
            Patch(facecolor=no_speech, label='Non-Speech', alpha=0.6),
            Patch(facecolor=highlight_color, label='Speech Leakage')
        ]
        ax.legend(handles=legend_elements, bbox_to_anchor=(1, 1), loc='upper left')

        plt.tight_layout()
        plt.show()